In [1]:
%load_ext autoreload
%autoreload 2
import helper_functions as hf
from imports import *
import importlib
from pathlib import Path

num_available_cpus = multiprocessing.cpu_count()
print("Number of available CPUs:", num_available_cpus)

torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device =", device)
torch.set_default_tensor_type('torch.cuda.FloatTensor') if torch.cuda.is_available() else print ('cpu')

torch.set_num_threads(num_available_cpus)

print("Number of threads:", torch.get_num_threads())
print("Number of interop threads:", torch.get_num_interop_threads())

/nobackup/users/sambt/anaconda3/envs/quak/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of available CPUs: 80
Device = cuda:0
Number of threads: 80
Number of interop threads: 80


# Evaluating against trainings on dataSB (2017)

In [9]:
year = 2017
signal_samples = ["Qstar2000_W400_UL17","Wp3000_B400_UL17","XYY_X3000_Y80_UL17"]

bkg_flowName = "dataSB_massFlat_{0}_clip10_NSRATQUAD_k6_hf120_nbpl4_tb10_pt300.pt".format(year)

sig_flowNames = [sig+"_dataSB_{0}_clip10_NSRATQUAD_k6_hf120_nbpl4_tb10_pt300.pt".format(year) for sig in signal_samples]

bkg_flow = hf.load_model(name=bkg_flowName)
sig_flows = [hf.load_model(name=signame) for signame in sig_flowNames]

# Make paths for saving output
Path("evaluations/dataSB_massFlat/{0}/nominal".format(year)).mkdir(parents=True,exist_ok=True)

In [10]:
# Evaluate on QCD background, normalized using dataSB mean/std
with h5py.File("bkgMeanStds/dataSB_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]
for i in range(36):
    norm, unnorm, mass = hf.load_bkg_batch_unnorm("QCDBKG",year,i,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_loss = -bkg_flow.eval_log_prob(norm)[0]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/dataSB_massFlat/{0}/nominal/eval_QCDBKG_{1}.h5".format(year,i),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        f.create_dataset("dataSB_massFlat_{0}".format(year),data=bkg_loss)
        for j in range(len(sig_losses)):
            f.create_dataset(signal_samples[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_loss, sig_losses

In [11]:
# Evaluate on signals, normalized using dataSB mean/std
with h5py.File("bkgMeanStds/dataSB_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]
for sig_samp in signal_samples:
    norm, unnorm, mass, varnames = hf.LAPS_train(sig_samp,year,num_batches=1,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_loss = -bkg_flow.eval_log_prob(norm)[0]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/dataSB_massFlat/{0}/nominal/eval_{1}.h5".format(year,sig_samp),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        f.create_dataset("dataSB_massFlat_{0}".format(year),data=bkg_loss)
        for j in range(len(sig_losses)):
            f.create_dataset(signal_samples[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_loss, sig_losses

# Evaluate against trainings on QCDBKG MC (2017)

In [2]:
year = 2017
signal_samples = ["Qstar2000_W400_UL17","Wp3000_B400_UL17","XYY_X3000_Y80_UL17"]
signal_flowNames = signal_samples + ["-and-".join(signal_samples)]

bkg_names = ["QCDBKG_massFlat_{0}".format(year)]
bkg_flowNames = [name+"_clip15_NSRATQUAD_k6_hf120_nbpl4_tb15_pt300.pt" for name in bkg_names]
bkg_flows = [hf.load_model(name=bkgname) for bkgname in bkg_flowNames]

sig_flowNames = [sig+"_QCDBKG_{0}_clip15_NSRATQUAD_k6_hf120_nbpl4_tb15_pt300.pt".format(year) for sig in signal_flowNames]
sig_flows = [hf.load_model(name=signame) for signame in sig_flowNames]

# Make paths for saving output
Path("evaluations/qcdbkg_massFlat/{0}/nominal".format(year)).mkdir(parents=True,exist_ok=True)

In [3]:
# Evaluate on QCD background, normalized using QCDBKG mean/std
with h5py.File("bkgMeanStds/QCDBKG_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]
for i in range(36):
    norm, unnorm, mass = hf.load_bkg_batch_unnorm("QCDBKG",year,i,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_losses = [-bflow.eval_log_prob(norm)[0] for bflow in bkg_flows]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/qcdbkg_massFlat/{0}/nominal/eval_QCDBKG_{1}.h5".format(year,i),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        for j in range(len(bkg_losses)):
            f.create_dataset(bkg_names[j],data=bkg_losses[j])
        for j in range(len(sig_losses)):
            f.create_dataset(signal_flowNames[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_losses, sig_losses

In [5]:
# Evaluate on signals, normalized using QCDBKG mean/std
with h5py.File("bkgMeanStds/QCDBKG_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]
for sig_samp in signal_samples:
    norm, unnorm, mass, varnames = hf.LAPS_train(sig_samp,year,num_batches=1,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_losses = [-bflow.eval_log_prob(norm)[0] for bflow in bkg_flows]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/qcdbkg_massFlat/{0}/nominal/eval_{1}.h5".format(year,sig_samp),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        for j in range(len(bkg_losses)):
            f.create_dataset(bkg_names[j],data=bkg_losses[j])
        for j in range(len(sig_losses)):
            f.create_dataset(signal_flowNames[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_losses, sig_losses

In [4]:
# Evaluate on some additional signal samples
extra_sigSamps = ["Wkk_W2000_R400_UL17","Wkk_W3000_R400_UL17","Wkk_W5000_R170_UL17","RSG_M3000_UL17","RSG_M5000_UL17"]

with h5py.File("bkgMeanStds/QCDBKG_{0}.h5".format(year)) as f:
    mean = f["mean"][()]
    std = f["std"][()]

for sig_samp in extra_sigSamps:
    norm, unnorm, mass, varnames = hf.LAPS_train(sig_samp,year,num_batches=1,Mjj_cut=800,pt_cut=300)
    norm = hf.normalize_data(norm,inp_mean=mean,inp_std=std)
    bkg_losses = [-bflow.eval_log_prob(norm)[0] for bflow in bkg_flows]
    sig_losses = [-sflow.eval_log_prob(norm)[0] for sflow in sig_flows]
    with h5py.File("evaluations/qcdbkg_massFlat/{0}/nominal/eval_{1}.h5".format(year,sig_samp),"w") as f:
        f.create_dataset("mass",data=mass[:,0])
        for j in range(len(bkg_losses)):
            f.create_dataset(bkg_names[j],data=bkg_losses[j])
        for j in range(len(sig_losses)):
            f.create_dataset(signal_flowNames[j],data=sig_losses[j])
    del norm, unnorm, mass, bkg_losses, sig_losses

# Test h5s

In [1]:
import h5py
fn = "evaluations/qcdbkg_massFlat/2017/nominal/eval_QCDBKG_0.h5"
f = h5py.File(fn,"r")

In [6]:
f.keys()

<KeysViewHDF5 ['QCDBKG_massFlat_2017', 'Qstar2000_W400_UL17', 'Wp3000_B400_UL17', 'XYY_X3000_Y80_UL17', 'mass']>

In [18]:
fn2 = "evaluations/qcdbkg_massFlat/2017/quakSpaces/QCDBKG_massFlat_2017-Qstar2000_W400_UL17/eval_QCDBKG_0.h5"
f2 = h5py.File(fn2,"r")

In [19]:
f2.keys()

<KeysViewHDF5 ['QCDBKG_massFlat_2017', 'Qstar2000_W400_UL17', 'mass']>

In [17]:
f['QCDBKG_massFlat_2017'][()]

array([11.942072, 18.57135 ,  9.34004 , ...,  7.553689,  8.994853,
        9.446538], dtype=float32)

In [16]:
f2['QCDBKG_massFlat_2017'][()]

array([ 9.309655 , 16.386232 ,  8.16436  , ...,  6.2197022,  7.7392936,
        8.300994 ], dtype=float32)